# Exercise 3 — Merge Word-Frequency Dictionaries

## Objective

Multiple servers analyze text independently and return dictionaries containing words and their frequencies.

The goal is to combine those dictionaries into a single dictionary where:

- every word appearing on any server is included;
- frequencies for the same word are added together;
- optionally, the result is sorted from highest frequency to lowest.

For example:

```python
d1 = {'python': 10, 'java': 3, 'c#': 8, 'javascript': 15}
d2 = {'java': 10, 'c++': 10, 'c#': 4, 'go': 9, 'python': 6}
d3 = {'erlang': 5, 'haskell': 2, 'python': 1, 'pascal': 1}
```

should produce totals such as:

```python
{
    'python': 17,
    'javascript': 15,
    'java': 13,
    'c#': 12,
    'c++': 10,
    'go': 9,
    'erlang': 5,
    'haskell': 2,
    'pascal': 1,
}
```

## Example server data

In [1]:
d1 = {
    'python': 10,
    'java': 3,
    'c#': 8,
    'javascript': 15,
}

d2 = {
    'java': 10,
    'c++': 10,
    'c#': 4,
    'go': 9,
    'python': 6,
}

d3 = {
    'erlang': 5,
    'haskell': 2,
    'python': 1,
    'pascal': 1,
}

---

# Core Pythonic Solution

A clean solution should work for any number of server dictionaries rather than being limited to exactly three.

The function below:

1. discovers every unique word using set union;
2. sums the frequency for that word across all supplied dictionaries;
3. uses `dict.get(word, 0)` so a missing word contributes zero.

In [2]:
def combine_frequencies(*frequency_maps):
    """Combine word frequencies from any number of dictionaries."""
    all_words = set().union(*frequency_maps)

    return {
        word: sum(mapping.get(word, 0) for mapping in frequency_maps)
        for word in all_words
    }

## Combine all three servers

In [3]:
combined = combine_frequencies(d1, d2, d3)
print(combined)

{'python': 17, 'pascal': 1, 'c#': 12, 'java': 13, 'javascript': 15, 'go': 9, 'haskell': 2, 'erlang': 5, 'c++': 10}


The frequencies are correct, although the ordering is not guaranteed because the implementation iterates over a set.

The next step adds deterministic sorting.

---

# Sort by Frequency

The exercise offers bonus points for sorting from highest frequency to lowest.

We can sort `(word, frequency)` pairs using:

```python
key=lambda item: item[1]
```

and `reverse=True`.

In [4]:
def sort_frequencies(frequencies):
    """Return a new dictionary sorted from highest to lowest frequency."""
    return {
        word: count
        for word, count in sorted(
            frequencies.items(),
            key=lambda item: item[1],
            reverse=True,
        )
    }

In [5]:
sorted_combined = sort_frequencies(
    combine_frequencies(d1, d2, d3)
)

print(sorted_combined)

{'python': 17, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10, 'go': 9, 'erlang': 5, 'haskell': 2, 'pascal': 1}


Expected result:

```text
{'python': 17, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10, 'go': 9, 'erlang': 5, 'haskell': 2, 'pascal': 1}
```

## Verify the two-server case

In [6]:
two_server_result = sort_frequencies(
    combine_frequencies(d1, d2)
)

print(two_server_result)

{'python': 16, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10, 'go': 9}


Expected result:

```text
{'python': 16, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10, 'go': 9}
```

---

# Recommended Solution — `collections.Counter`

Python's standard library already provides a specialized type for frequency data: `collections.Counter`.

`Counter` behaves similarly to a dictionary, but it is specifically designed for counting hashable objects.

One especially useful property is that counters can be added together:

```python
Counter({'python': 10}) + Counter({'python': 6})
```

produces a counter where `python` has frequency `16`.

For real frequency-analysis code, `Counter` is usually the most expressive solution.

In [7]:
from collections import Counter


def combine_frequencies_counter(*frequency_maps):
    """Combine frequency mappings using collections.Counter."""
    total = Counter()

    for mapping in frequency_maps:
        total.update(mapping)

    return dict(total)

In [8]:
counter_result = combine_frequencies_counter(d1, d2, d3)
print(counter_result)

{'python': 17, 'java': 13, 'c#': 12, 'javascript': 15, 'c++': 10, 'go': 9, 'erlang': 5, 'haskell': 2, 'pascal': 1}


`Counter.update()` adds counts rather than replacing them, which makes it particularly suitable for aggregating frequency dictionaries.

---

# Production-Quality Implementation

The following implementation adds several practical features:

- accepts any number of mappings;
- accepts general mapping objects, not only dictionaries;
- returns deterministic output;
- supports ascending or descending order;
- supports optional alphabetical tie-breaking;
- supports optional minimum-frequency filtering;
- supports optional top-N limiting;
- validates frequency values;
- does not mutate the input mappings.

In [9]:
from collections import Counter
from collections.abc import Mapping
from numbers import Real
from typing import Hashable


def aggregate_frequencies(
    *frequency_maps: Mapping[Hashable, Real],
    descending: bool = True,
    alphabetical_ties: bool = True,
    min_frequency: Real | None = None,
    top_n: int | None = None,
) -> dict[Hashable, Real]:
    """Aggregate frequencies from multiple mappings.

    Parameters
    ----------
    *frequency_maps:
        Zero or more mappings containing item -> frequency pairs.
    descending:
        Sort largest frequencies first when True.
    alphabetical_ties:
        For string keys with equal frequencies, sort alphabetically.
        Non-string keys use their string representation for deterministic
        tie-breaking.
    min_frequency:
        If supplied, exclude entries whose total frequency is below this
        value.
    top_n:
        If supplied, return at most this many entries after sorting and
        filtering.

    Returns
    -------
    dict
        A new dictionary containing aggregated frequencies.

    Raises
    ------
    TypeError
        If an input is not a Mapping or a frequency is not numeric.
    ValueError
        If a frequency is negative or top_n is negative.

    Notes
    -----
    Input mappings are never modified.
    """
    if top_n is not None:
        if not isinstance(top_n, int):
            raise TypeError('top_n must be an integer or None.')
        if top_n < 0:
            raise ValueError('top_n cannot be negative.')

    totals = Counter()

    for index, mapping in enumerate(frequency_maps, start=1):
        if not isinstance(mapping, Mapping):
            raise TypeError(
                f'Argument {index} must be a mapping; '
                f'got {type(mapping).__name__}.'
            )

        for item, frequency in mapping.items():
            if isinstance(frequency, bool) or not isinstance(frequency, Real):
                raise TypeError(
                    f'Frequency for {item!r} must be numeric; '
                    f'got {type(frequency).__name__}.'
                )

            if frequency < 0:
                raise ValueError(
                    f'Frequency for {item!r} cannot be negative.'
                )

            totals[item] += frequency

    items = totals.items()

    if min_frequency is not None:
        items = (
            (item, frequency)
            for item, frequency in items
            if frequency >= min_frequency
        )

    items = list(items)

    if alphabetical_ties:
        if descending:
            items.sort(
                key=lambda pair: (-pair[1], str(pair[0]))
            )
        else:
            items.sort(
                key=lambda pair: (pair[1], str(pair[0]))
            )
    else:
        items.sort(
            key=lambda pair: pair[1],
            reverse=descending,
        )

    if top_n is not None:
        items = items[:top_n]

    return dict(items)

## Run the enhanced implementation

In [10]:
result = aggregate_frequencies(d1, d2, d3)
print(result)

{'python': 17, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10, 'go': 9, 'erlang': 5, 'haskell': 2, 'pascal': 1}


Expected output:

```text
{'python': 17, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10, 'go': 9, 'erlang': 5, 'haskell': 2, 'pascal': 1}
```

---

# Deterministic Tie-Breaking

Several entries can have the same frequency.

For reproducible output, the enhanced function sorts equal-frequency string keys alphabetically.

In [11]:
ties_1 = {
    'beta': 5,
    'alpha': 5,
}

ties_2 = {
    'gamma': 5,
}

tie_result = aggregate_frequencies(ties_1, ties_2)
print(tie_result)

{'alpha': 5, 'beta': 5, 'gamma': 5}


Expected output:

```text
{'alpha': 5, 'beta': 5, 'gamma': 5}
```

---

# Added Value: Top-N Frequencies

In real text-analysis workloads, we often care only about the most frequent terms.

The `top_n` option allows us to return only the highest-frequency items.

In [12]:
top_3 = aggregate_frequencies(
    d1,
    d2,
    d3,
    top_n=3,
)

print(top_3)

{'python': 17, 'javascript': 15, 'java': 13}


Expected output:

```text
{'python': 17, 'javascript': 15, 'java': 13}
```

---

# Added Value: Minimum-Frequency Filtering

Very uncommon words may be irrelevant for some analyses.

The `min_frequency` argument can remove low-frequency entries after aggregation.

In [13]:
frequent_only = aggregate_frequencies(
    d1,
    d2,
    d3,
    min_frequency=10,
)

print(frequent_only)

{'python': 17, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10}


Expected output:

```text
{'python': 17, 'javascript': 15, 'java': 13, 'c#': 12, 'c++': 10}
```

---

# Added Value: Combine Filtering and Top-N

In [14]:
filtered_top = aggregate_frequencies(
    d1,
    d2,
    d3,
    min_frequency=10,
    top_n=3,
)

print(filtered_top)

{'python': 17, 'javascript': 15, 'java': 13}


Expected output:

```text
{'python': 17, 'javascript': 15, 'java': 13}
```

---

# Added Value: Ascending Order

Although the exercise requests highest-to-lowest sorting as a bonus, exposing the opposite ordering makes the function reusable.

In [15]:
ascending = aggregate_frequencies(
    d1,
    d2,
    d3,
    descending=False,
)

print(ascending)

{'pascal': 1, 'haskell': 2, 'erlang': 5, 'go': 9, 'c++': 10, 'c#': 12, 'java': 13, 'javascript': 15, 'python': 17}


Because `haskell` and `pascal`/other values can tie at small frequencies, deterministic tie-breaking makes the result reproducible.

---

# Verification Against the Exercise Requirements

In [16]:
expected_all_servers = {
    'python': 17,
    'javascript': 15,
    'java': 13,
    'c#': 12,
    'c++': 10,
    'go': 9,
    'erlang': 5,
    'haskell': 2,
    'pascal': 1,
}

expected_two_servers = {
    'python': 16,
    'javascript': 15,
    'java': 13,
    'c#': 12,
    'c++': 10,
    'go': 9,
}

assert aggregate_frequencies(d1, d2, d3) == expected_all_servers
assert aggregate_frequencies(d1, d2) == expected_two_servers

print('Exercise requirement tests passed.')

Exercise requirement tests passed.


---

# Edge-Case Tests

In [17]:
# No servers
assert aggregate_frequencies() == {}

# One empty server
assert aggregate_frequencies({}) == {}

# Multiple empty servers
assert aggregate_frequencies({}, {}, {}) == {}

# One server only
assert aggregate_frequencies({'python': 5}) == {'python': 5}

# No overlapping words
assert aggregate_frequencies(
    {'python': 2},
    {'java': 3},
) == {
    'java': 3,
    'python': 2,
}

# All servers contain the same word
assert aggregate_frequencies(
    {'python': 1},
    {'python': 2},
    {'python': 3},
    {'python': 4},
) == {
    'python': 10,
}

# Zero counts are allowed
assert aggregate_frequencies(
    {'python': 0, 'java': 2},
    {'python': 5},
) == {
    'python': 5,
    'java': 2,
}

# Filtering
assert aggregate_frequencies(
    {'a': 1, 'b': 10},
    {'a': 2, 'b': 5},
    min_frequency=5,
) == {
    'b': 15,
}

# top_n=0 intentionally returns an empty result
assert aggregate_frequencies(
    {'a': 10, 'b': 20},
    top_n=0,
) == {}

print('All edge-case tests passed.')

All edge-case tests passed.


---

# Validation Tests

Frequency data normally represents counts, so negative values and non-numeric counts are usually evidence of invalid input.

The enhanced function catches these mistakes early.

In [18]:
def assert_raises(expected_exception, function, *args, **kwargs):
    """Small helper for exception tests without external libraries."""
    try:
        function(*args, **kwargs)
    except expected_exception:
        return
    except Exception as exc:
        raise AssertionError(
            f'Expected {expected_exception.__name__}, '
            f'but got {type(exc).__name__}.'
        ) from exc

    raise AssertionError(
        f'Expected {expected_exception.__name__} to be raised.'
    )

In [19]:
assert_raises(
    TypeError,
    aggregate_frequencies,
    ['not', 'a', 'mapping'],
)

assert_raises(
    TypeError,
    aggregate_frequencies,
    {'python': 'ten'},
)

assert_raises(
    ValueError,
    aggregate_frequencies,
    {'python': -1},
)

assert_raises(
    ValueError,
    aggregate_frequencies,
    {'python': 10},
    top_n=-1,
)

assert_raises(
    TypeError,
    aggregate_frequencies,
    {'python': 10},
    top_n=2.5,
)

print('Validation tests passed.')

Validation tests passed.


---

# Verify Inputs Are Not Modified

In [20]:
d1_before = d1.copy()
d2_before = d2.copy()
d3_before = d3.copy()

_ = aggregate_frequencies(d1, d2, d3)

assert d1 == d1_before
assert d2 == d2_before
assert d3 == d3_before

print('Input dictionaries were not modified.')

Input dictionaries were not modified.


---

# Understanding `Counter.update()`

`Counter.update()` differs from ordinary `dict.update()`.

A normal dictionary update replaces an existing value:

```python
{'python': 10}.update({'python': 6})
```

would leave `python` with `6`.

`Counter.update()` instead **adds** the supplied count.

That is exactly the behavior required for this exercise.

In [21]:
demo = Counter({'python': 10})
demo.update({'python': 6, 'java': 3})

print(demo)

Counter({'python': 16, 'java': 3})


Expected output:

```text
Counter({'python': 16, 'java': 3})
```

---

# Alternative: `Counter` Addition

`Counter` instances also support addition:

In [22]:
combined_with_addition = Counter(d1) + Counter(d2) + Counter(d3)

print(dict(combined_with_addition))

{'python': 17, 'java': 13, 'c#': 12, 'javascript': 15, 'c++': 10, 'go': 9, 'erlang': 5, 'haskell': 2, 'pascal': 1}


This is concise and readable for a known, small number of dictionaries.

For an arbitrary number of mappings, repeatedly calling `Counter.update()` is usually clearer and avoids constructing a long addition expression.

---

# Important `Counter` Detail

`Counter` addition has special semantics: zero and negative results are omitted.

For ordinary word-frequency data this is normally desirable because counts should be non-negative.

However, it is one reason the enhanced implementation explicitly validates input frequencies rather than silently accepting arbitrary negative values.

---

# Complexity Analysis

Let:

- **S** be the number of servers;
- **N** be the total number of `(word, frequency)` entries across all server dictionaries;
- **U** be the number of unique words after aggregation.

## Aggregation

Each input entry needs to be examined once.

Dictionary/Counter lookup and update operations are `O(1)` on average.

Therefore aggregation is approximately:

**Time:** `O(N)` average-case

**Space:** `O(U)`

## Sorting

Sorting the `U` unique words requires:

**Time:** `O(U log U)`

So the complete sorted solution is:

**Time:** `O(N + U log U)`

**Space:** `O(U)`

---

# Performance Consideration for Very Large Systems

For a small number of server dictionaries, sorting every combined item is perfectly reasonable.

If there are millions of unique words and only the top few are required, sorting the entire result can do unnecessary work.

`Counter.most_common(n)` is useful in that situation because it directly expresses the top-N frequency operation.

In [23]:
def top_frequencies(*frequency_maps, n=10):
    """Return the n most common aggregated items."""
    if not isinstance(n, int):
        raise TypeError('n must be an integer.')

    if n < 0:
        raise ValueError('n cannot be negative.')

    totals = Counter()

    for mapping in frequency_maps:
        totals.update(mapping)

    return dict(totals.most_common(n))

In [24]:
print(top_frequencies(d1, d2, d3, n=3))

{'python': 17, 'javascript': 15, 'java': 13}


Expected output:

```text
{'python': 17, 'javascript': 15, 'java': 13}
```

---

# Practical Interpretation

This exercise models a common distributed-data pattern:

```text
server 1 ─┐
server 2 ─┼──> aggregate counts ──> sort/filter ──> result
server 3 ─┘
```

The same approach can be applied to:

- word counts;
- API endpoint usage;
- HTTP status-code counts;
- product purchase counts;
- search-query frequencies;
- event counts;
- log-message categories;
- distributed metrics.

The important idea is that each server produces a partial aggregation and those partial aggregations are then reduced into one global result.

---

# Final Answer

For the original exercise, a concise Pythonic solution is:

```python
def combine_frequencies(*frequency_maps):
    all_words = set().union(*frequency_maps)

    totals = {
        word: sum(mapping.get(word, 0) for mapping in frequency_maps)
        for word in all_words
    }

    return dict(
        sorted(
            totals.items(),
            key=lambda item: item[1],
            reverse=True,
        )
    )
```

For real frequency-processing code, the standard-library `Counter` approach is generally preferable:

```python
from collections import Counter


def combine_frequencies(*frequency_maps):
    total = Counter()

    for mapping in frequency_maps:
        total.update(mapping)

    return dict(total.most_common())
```

This version is concise, expressive, works with any number of servers, and directly uses a Python standard-library abstraction designed for frequency data.

## Summary

Key ideas demonstrated in this exercise:

- dictionary/set union for discovering all keys;
- `dict.get(key, 0)` for missing-value defaults;
- generator expressions with `sum()`;
- sorting dictionaries by value;
- `collections.Counter` for frequency aggregation;
- arbitrary numbers of input dictionaries with `*args`;
- deterministic tie-breaking;
- top-N analysis;
- minimum-frequency filtering;
- defensive validation;
- non-mutating API design;
- complexity analysis for distributed aggregation.